# GNN-QLSTM Benchmark — Colab

**Thứ tự chạy:**
1. Cell 1 — Install → **Restart runtime**
2. Cell 2 — Upload + Extract zip
3. Cell 3 — Upload checkpoints zip
4. Cell 4 — Mount Drive
5. Cell 5 — Chạy benchmark
6. Cell 6 — Download kết quả

In [ ]:
# Cell 1 — Install (sau khi xong: Runtime → Restart runtime)
import torch

!pip install -q pennylane pennylane-lightning networkx matplotlib
!pip install -q torch_geometric

TORCH = torch.__version__.split('+')[0]
if torch.cuda.is_available() and torch.version.cuda:
    CUDA = 'cu' + torch.version.cuda.replace('.', '')
    !pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
else:
    !pip install -q torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-{TORCH}+cpu.html

print('Done! >>> Restart runtime now.')

In [ ]:
# Cell 2 — Upload QLSTM_with_GNN.zip + Extract
from google.colab import files
import zipfile, os, sys

uploaded = files.upload()  # chọn QLSTM_with_GNN.zip

with zipfile.ZipFile('QLSTM_with_GNN.zip', 'r') as z:
    z.extractall('/content/')

os.chdir('/content')
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('data', exist_ok=True)

sys.path.insert(0, '/content')
sys.path.insert(0, '/content/benchmark')

print('Files:', [f for f in sorted(os.listdir('.')) if not f.startswith('.')])
print('Data: ', sorted(os.listdir('data')))

In [ ]:
# Cell 3 — Upload checkpoints_p2.zip
from google.colab import files
import zipfile

uploaded = files.upload()  # chọn checkpoints_p2.zip

with zipfile.ZipFile('checkpoints_p2.zip', 'r') as z:
    z.extractall('checkpoints/')

print('Checkpoints:', sorted(os.listdir('checkpoints')))

In [ ]:
# Cell 4 — Mount Google Drive (lưu kết quả vĩnh viễn)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_RESULTS = '/content/drive/MyDrive/QLSTM_benchmark_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)

local_results = '/content/benchmark_results'
if os.path.exists(local_results) and not os.path.islink(local_results):
    for f in os.listdir(local_results):
        shutil.move(os.path.join(local_results, f), DRIVE_RESULTS)
    os.rmdir(local_results)
if not os.path.exists(local_results):
    os.symlink(DRIVE_RESULTS, local_results)

existing = sorted(os.listdir('benchmark_results'))
print(f'benchmark_results/ → {DRIVE_RESULTS}')
print(f'Existing results: {existing if existing else "(trống — chạy mới)"}')

In [ ]:
# Cell 5 — Chạy benchmark (N_INSTANCES=5, ~30-60 phút)
import importlib.util, sys

# Load benchmark.py trực tiếp (tránh conflict với tên thư mục)
spec = importlib.util.spec_from_file_location("bm", "/content/benchmark/benchmark.py")
bm = importlib.util.module_from_spec(spec)
sys.modules["bm"] = bm
spec.loader.exec_module(bm)

bm.N_INSTANCES = 5   # giảm từ 20 → 5 (4x nhanh hơn)

print(f'N_INSTANCES = {bm.N_INSTANCES}')
print(f'N_VALUES    = {bm.N_VALUES}')
print(f'T_INFERENCE = {bm.T_INFERENCE}')
print(f'QAOA_SIM    = {bm.QAOA_SIM}')
print()

# Benchmark cả QLSTM + CLSTM
sys.argv = ['benchmark.py']
bm.main()

In [ ]:
# Cell 6 — Download kết quả về máy
import shutil
from google.colab import files

shutil.make_archive('benchmark_results', 'zip', 'benchmark_results')
files.download('benchmark_results.zip')
print('Downloaded benchmark_results.zip')